# CancerEmo Dataset Exploration

## 1  Overview

This notebook explores the CancerEmo dataset before evaluating pre-trained text-emotion models. CancerEmo is a fine-grained emotion dataset collected from online discussion forums for cancer, where sentences are annotated with the presence of eight emotions [1].

The dataset is released as eight separate files, one for each emotion. In the files, a binary label is used (1 = emotion present, 0 = emotion absent). This exploration is a step toward understanding the data before its use, and it looks at the file structure, the predefined dataset splits, the class distribution and the data quality missing values, empty text and duplicate sentences.

CancerEmo is only used for evaluation because the project uses pre-trained models and does no training or fine-tuning. Thus, the original files are not touched and the only output of this notebook is a clean split of the validation and testing records that are saved for later use.

## 2  Importing Libraries

For this exploration we only need two libraries. Pathlib to locate the dataset files in a platform independent manner, and pandas, to load the CSV data files and check on their columns, records and labels.

In [1]:
# Import the required libraries
from pathlib import Path
import pandas as pd

## 3  Loading the Dataset

There are eight CancerEmo files, each corresponding to one emotion. So, the files are loaded independently and separated at this stage. This allows checking of the columns, record count and binary labels of each file before the data is prepared for evaluation.

In [2]:
# Folder for the raw CancerEmo CSV files
path = Path("../../data/raw/canceremo")

In [3]:
# Map each emotion to corresponding CSV file name
files = {
    "Anger": "Anger_anon.csv",
    "Anticipation": "Anticipation_anon.csv",
    "Disgust": "Disgust_anon.csv",
    "Fear": "Fear_anon.csv",
    "Joy": "Joy_anon.csv",
    "Sadness": "Sadness_anon.csv",
    "Surprise": "Surprise_anon.csv",
    "Trust": "Trust_anon.csv"
}

In [4]:
# loaded DataFrame by its emotion name
emotion_datasets = {}

In [5]:
# Load every emotion file and report how many records each one have
for emotion, filename in files.items():
    # file path
    file_path = path / filename
    # emotion dataset
    emotion_datasets[emotion] = pd.read_csv(file_path)
    # print statement
    print(f"{emotion}: {len(emotion_datasets[emotion])} records")

Anger: 837 records
Anticipation: 436 records
Disgust: 891 records
Fear: 5388 records
Joy: 6043 records
Sadness: 3606 records
Surprise: 826 records
Trust: 1887 records


To confirm that the sentence, the emotion label and the predefined split column were all loaded correctly, the first five rows of each file are displayed below.

In [6]:
# first five rows of every emotion file
for emotion, dataset in emotion_datasets.items():
    display(dataset.head(5))

,Sentence,Anger,Split
0,And it will no doubt make me happy in the morn...,0,0
1,My dr gave me a rx for morphine oral to take f...,0,0
2,surgeon did broncos copy on upper lobe of left...,0,0
3,The apex seems free of trouble.,0,0
4,Good morning..I have been reading this post an...,0,0


,Sentence,Anticipation,Split
0,Isn't it wonderful to have something to rejoic...,0,0
1,I dont have lymphadema but I do get PT.,0,0
2,I am going to have the radiation was just conf...,0,0
3,I have met with my pulmonologist several times...,0,0
4,The procedure sounds very promising with suppo...,0,0


,Sentence,Disgust,Split
0,I would like to know if anyone else has had an...,0,0
1,"Hey <PERSON>, When I was on Taxotere everythin...",0,0
2,He has planned on 6 rounds prior to maintenanc...,0,0
3,Unfortunately the chemo hasn't worked and the ...,0,0
4,Enjoy whatever foods you want now!,0,0


,Sentence,Fear,Split
0,Hopefully we have more good days than we have ...,0,0
1,I explained that my own self esteem would be c...,0,0
2,We've been told that some physicians may take ...,0,0
3,They keep coming in every hour or so to do neu...,0,0
4,It has proven effective and improves the quali...,0,0


,Sentence,Joy,Split
0,The university teaching hospitals with cancer ...,0,0
1,We've been told that some physicians may take ...,0,0
2,With reference to MUSE - my question was : Doe...,0,0
3,"However, new symptoms have doctors insisting i...",0,0
4,"I wonder, where do you live?",0,0


,Sentence,Sadness,Split
0,May all your wishes come true and may you have...,0,0
1,With reference to MUSE - my question was : Doe...,0,0
2,They keep coming in every hour or so to do neu...,0,0
3,My wife is the one with stage 4 lung cancer an...,0,0
4,"However, it will be a few days before we get t...",0,0


,Sentence,Surprise,Split
0,I even watched how to make a scarf out of a te...,0,0
1,I believe the scoring is from 0 to 6.,0,0
2,I was advised to take 1 Claritin and 2 Tyleno...,0,0
3,Hope your day is filled with joy and happiness...,0,0
4,"I wonder, where do you live?",0,0


,Sentence,Trust,Split
0,My body is sick of being sick and tired.,0,0
1,I think I like the 4 hour one the best!,0,0
2,All I knew about cancer when I was first diagn...,0,0
3,"BETWEEN THE TWO CANCERS, I AM SO CONFUSED AND ...",0,0
4,The radiologist said due to the pattern it did...,0,0


All 8 CancerEmo files loaded successfully. Each record has a sentence, a binary label for the emotion of this file, and a predefined split value.

**Analysis:** 

The size of the eight files varies significantly, which is an important aspect to consider before to assessment. Joy (6,043 records) and Fear (5,388 records) are the biggest files, followed by Sadness (3,606) and Trust (1,887), Anticipation (436), Surprise (826), Anger (837), and Disgust (891) are the smallest. A single overall accuracy figure across all emotions would be dominated by the two large classes (joy and fear) and would effectively hide the performance on the tiny classes (anticipation) due to the nearly ten-fold gap between the largest and smallest emotion. This is the first piece of evidence supporting the implementation of a per-emotion, macro-averaged evaluation later on in the project, in which all eight emotions, regardless of the number of recordings they include, contribute equally to the final score. Since the dataset is only used for evaluation, the different sizes are a feature of the original dataset and are maintained rather than changed.

## 4  Understanding the Dataset Split

CancerEmo already uses the Split column to divide its data into training, validation, and testing sets, with 0 = training, 1 = validation, and 2 = testing. To ensure that the evaluation data is never combined with other records, which could risk a fair comparison, these initial divides are reviewed and kept.

In [7]:
# per-emotion split counts
split_results = []

In [8]:
# training, validation and testing records each emotion file has
for emotion, dataset in emotion_datasets.items():
    # split counts
    split_counts = dataset["Split"].value_counts()
    # append to the list
    split_results.append({
        "Emotion": emotion,
        "Training": split_counts.get(0, 0),
        "Validation": split_counts.get(1, 0),
        "Testing": split_counts.get(2, 0)
    })

In [9]:
# split distribution as dataframe
df = pd.DataFrame(split_results)
df

,Emotion,Training,Validation,Testing
0,Anger,669,84,84
1,Anticipation,360,34,42
2,Disgust,735,90,66
3,Fear,4310,539,539
4,Joy,4834,604,605
5,Sadness,2884,361,361
6,Surprise,614,102,110
7,Trust,1509,189,189


**Analysis:** 

The table verifies that each emotion file follows to the intended split structure of the dataset, with the majority of records allocated for training and significantly smaller amounts put aside for testing and validation. For example, there are 4,310 training, 539 validation, and 539 testing records for Fear and 4,834 training, 604 validation, and 605 testing data for Joy. In contrast, the smallest emotion, Anticipation, has just 360 training, 34 validation, and 42 testing records. This leads to two methodological points.

First, the amount of samples accessible for evaluation is restricted, particularly for the smaller emotions Anticipation contains just 34 validation and 42 testing words, because the validation and testing sets are purposefully small in comparison to training. Any evaluation conclusions on these small emotions must therefore be treated as indicative rather than statistically strong, and this is acknowledged as a limitation.

Second and most crucially, these predetermined splits are never combined and are preserved exactly as provided. The training data are presented here simply for completeness and are not used at all because this project assesses pre-trained models. The testing split is held back and used only once, for the final assessment of the one selected model, while the validation split is used for evaluating the candidate pre-trained models against one another model selection. The model is chosen on one set of data and reported on another when these two assessment sets are kept completely separate, preventing the selection process from optimistically biasing the final reported outcome.

## 5  Class (Label) Distribution

In each file, a sentence is labeled 1 when the emotion is present and 0 when it is not. The amount of positive and negative records is verified in both the validation and testing splits since a significant difference between positive and negative labels might skew assessment metrics.

In [10]:
# positive and negative label counts per emotion
label_distribution = []

In [11]:
# For each emotion count positive and negative labels in validation and testing splits
for emotion, dataset in emotion_datasets.items():
    # validation data
    validation_data = dataset[dataset["Split"] == 1]
    # testing data
    testing_data = dataset[dataset["Split"] == 2]
    # label distributions
    label_distribution.append({
        "Emotion": emotion,
        "Validation Negative": (validation_data[emotion] == 0).sum(),
        "Validation Positive": (validation_data[emotion] == 1).sum(),
        "Testing Negative": (testing_data[emotion] == 0).sum(),
        "Testing Positive": (testing_data[emotion] == 1).sum()
    })

In [12]:
# label distribution as dataframe
df_label = pd.DataFrame(label_distribution)
df_label

,Emotion,Validation Negative,Validation Positive,Testing Negative,Testing Positive
0,Anger,43,41,50,34
1,Anticipation,17,17,21,21
2,Disgust,45,45,33,33
3,Fear,279,260,253,286
4,Joy,293,311,304,301
5,Sadness,165,196,181,180
6,Surprise,51,51,55,55
7,Trust,94,95,96,93


**Analysis:** 

The positive and negative labels are nearly equal in both evaluation splits for each individual emotion file. Anger has 43 negative against 41 positive, Disgust 45 against 45, Fear 279 against 260, Joy 293 against 311, Sadness 165 against 196, Surprise 51 against 51, Trust 94 against 95, and Anticipation 17 against 17 in the validation divide. The testing split shows the same trend, with Joy at 304 against 301 and Fear at 253 negative vs 286 positive. No emotion comes close to the kind of significant skew like 90% of a class that would allow a model to achieve high scores just by consistently guessing the majority label.

This near-balance has two implications for the evaluation design and is a favorable and unusual characteristic. This means that no resampling, oversampling, or class re-balancing is required, which prevents any artificial modification from being introduced into a project that is solely intended to assess models and maintains the assessment faithful to the dataset as presented. Additionally, with balanced classes, accuracy is not misleading as it would be on a highly skewed dataset, so accuracy is a reasonable headline metric here. However, accuracy is not used alone to prevent any residual imbalance and to maintain the rigor of the comparison macro-F1, precision, and recall are given with accuracy, so a model is rewarded for correctly identifying the positive emotion-present events rather than only the negatives.

## 6  Checking Text and Label Quality

Each emotion file is examined for data-quality problems such as missing sentences, empty sentences, invalid binary labels, invalid split values, and duplicate sentences that could skew the evaluation. These tests verify that repetitive or incomplete records do not impact the text-model evaluation.

In [13]:
# data quality check results per emotion
checks = []

In [14]:
# data quality checks on each emotion file
for emotion, dataset in emotion_datasets.items():
    # no sentence at all
    missing_sentences = dataset["Sentence"].isnull().sum()
    # sentence is blank once whitespace is stripped
    empty_sentences = dataset["Sentence"].fillna("").str.strip().eq("").sum()
    # invalid labels and splits
    invalid_labels = (~dataset[emotion].isin([0, 1])).sum()
    invalid_splits = (~dataset["Split"].isin([0, 1, 2])).sum()
    # Repeated sentences
    duplicated_sentences = dataset["Sentence"].duplicated().sum()
    # append to the list
    checks.append({
        "Emotion": emotion,
        "Missing Sentences": missing_sentences,
        "Empty Sentences": empty_sentences,
        "Invalid Labels": invalid_labels,
        "Invalid Splits": invalid_splits,
        "Duplicated Sentences": duplicated_sentences
    })

In [15]:
# data quality checks as dataframe
df_checks = pd.DataFrame(checks)
df_checks

,Emotion,Missing Sentences,Empty Sentences,Invalid Labels,Invalid Splits,Duplicated Sentences
0,Anger,0,0,0,0,0
1,Anticipation,0,0,0,0,0
2,Disgust,0,0,0,0,0
3,Fear,0,0,0,0,0
4,Joy,0,0,0,0,0
5,Sadness,0,0,0,0,0
6,Surprise,0,0,0,0,0
7,Trust,0,0,0,0,0


**Analysis:** 

For each of the eight emotion files, the quality checks give zero for every category no duplicate sentences, no labels outside of the valid 0, 1 set, no split values outside of 0, 1, 2, and no missing or empty phrases. Each of these problems would otherwise compromise the evaluation in a certain way, thus this is a significant and comforting outcome. Duplicate sentences would allow the same example to be counted more than once, distorting the metrics and, if duplicates crossed the validation and testing sets, leaking information between model selection and final evaluation
missing or empty sentences would give the models no text to evaluate and would produce meaningless predictions and invalid labels or split values would corrupt the ground truth against which the models are scored.

The original CancerEmo files can be used exactly as published because all five checks are passed, meaning that no records need to be removed, cleaned, or altered. This directly supports the project's methodology there is no chance that the author will accidentally change the data in a way that would skew the results because the dataset is handled just as an unmodified evaluation benchmark and there is no cleaning phase.

Anger, anticipation, disgust, fear, joy, sadness, surprise, and trust are among the eight independent binary classification tasks that CancerEmo creates, with 1 indicating the presence of an emotion and 0 indicating its absence. The testing split is set aside for the final assessment of the chosen model, while the validation split is used to compare potential models. The primary comparison metric is the mean macro-F1 over the eight tasks.

## 7  Preparing the Evaluation Splits

For the purpose of evaluating the model later, the testing and validation records are kept apart. While the validation data is used to compare the potential pre-trained text models, the testing data is saved for the final assessment of the selected model. The training records are not used because no training or fine-tuning is carried out.

In [16]:
# validation and testing subsets for each emotion
validation_datasets = {}
testing_datasets = {}

In [17]:
# size of each evaluation split for reporting
evaluationsplit = []

In [18]:
# validation Split = 1 and testing Split = 2 rows for each emotion
for emotion, dataset in emotion_datasets.items():
    # validation dataset
    validation_datasets[emotion] = (dataset.loc[dataset["Split"] == 1, ["Sentence", emotion]].reset_index(drop=True))
    # testing dataset
    testing_datasets[emotion] = (dataset.loc[dataset["Split"] == 2, ["Sentence", emotion]].reset_index(drop=True))
    # evaluation split append
    evaluationsplit.append({
        "Emotion": emotion,
        "Validation Records": len(validation_datasets[emotion]),
        "Testing Records": len(testing_datasets[emotion])
    })

In [19]:
# evaluation split sizes as dataframe
df_evaluation = pd.DataFrame(evaluationsplit)
df_evaluation

,Emotion,Validation Records,Testing Records
0,Anger,84,84
1,Anticipation,34,42
2,Disgust,90,66
3,Fear,539,539
4,Joy,604,605
5,Sadness,361,361
6,Surprise,102,110
7,Trust,189,189


**Analysis:** 

The validation and testing counts found earlier in Section 4 are precisely matched by the extracted evaluation sets. For example, Fear yields 539 validation and 539 testing records, Joy 604 and 605, and Anticipation 34 and 42, indicating that the extraction selected the right rows and that no records were lost or duplicated during the process. Only the two columns required for evaluation the Sentence and its binary label for that emotion remain in each extracted record. In keeping with the pre-trained, no fine-tuning method, the training rows Split = 0 are purposefully left out.

The subsequent model evaluation notebooks are made simpler and less error prone by maintaining the validation and testing sets as separate, clearly labelled collections at this point. This is because the model selection code loads only the validation files, and the final evaluation code loads only the testing files, preventing the two roles from being accidentally confused.

## 8 Save the Evaluation Split

CSV files now contain the prepared validation and testing records. By saving them, the original CancerEmo files are preserved and this preparation is not repeated each time the candidate text-emotion models are assessed.

In [20]:
# processed split evaluation files
processed_path = Path("../../data/processed/canceremo")

In [21]:
# sub-folders keep the model selection data apart from the final evaluation data
validation_folder = processed_path / "validation"
testing_folder = processed_path / "testing"

In [22]:
# folders if they do not already exist
validation_folder.mkdir(parents=True, exist_ok=True)
testing_folder.mkdir(parents=True, exist_ok=True)

In [23]:
# validation CSV and testing CSV per emotion
for emotion in emotion_datasets:
    # file name
    filename = f"{emotion.lower()}.csv"
    # validaiton data
    validation_datasets[emotion].to_csv(validation_folder / filename, index=False)
    # testing data
    testing_datasets[emotion].to_csv(testing_folder / filename, index=False)

In [24]:
# all eight validation and testing files were saved
print("Validation files:", len(validation_datasets))
print("Testing files:", len(testing_datasets))

Validation files: 8
Testing files: 8


**Analysis:** 

All eight emotions were successfully saved into their corresponding folders, with one CSV for each emotion, according to the confirmation that 8 validation files and 8 testing files were produced. This results in a final evaluation that differentiates the two evaluation responsibilities at the file-system level and provides a clear, repeatable data layout validation for model selection and testing. Importantly, the exploration is completely repeatable and the benchmark is preserved because only the processed split files are produced the original raw CancerEmo files are never changed.

## 9  Conclusion

The CancerEmo dataset was analysed for each of the eight emotion categories anger, anticipation, disgust, fear, joy, sadness, surprise, and trust. The validation and testing splits revealed a nearly balanced distribution of positive and negative labels within each emotion, and there were no missing sentences, empty text, invalid labels, invalid split values, or duplicate phrases in the data.

The investigation also revealed a significant difference in file size among a number of emotions from 436 records for Anticipation to 6,043 for Joy, and as a result, the assessment sets for the less common emotions were smaller. These findings directly influenced the evaluation strategy no resampling was applied because the per-class label balance is good, and the models are compared using per-emotion metrics averaged with macro-F1 because per-emotion sample sizes vary so much that each emotion counts equally and performance on the small classes is not masked by the large ones.

Thus, there was no need for text editing, cleaning, or balancing. The original dataset divides were maintained the validation data was prepared for comparing potential pre-trained text-emotion models, while the testing data was reserved for the final assessment of the selected model. The processed records were saved into distinct validation and testing folders in preparation for the model-evaluation phase, while the original CancerEmo files remain unchanged.

## References

[1] T. Sosea and C. Caragea, "CancerEmo: A dataset for fine-grained emotion detection," in Proc. Conf. Empirical Methods in Natural Language Processing (EMNLP), Online, 2020, pp. 8892–8904, doi: 10.18653/v1/2020.emnlp-main.715.